In [1]:
import pandas as pd

# define important cols to speed up the read process
keep_cols = ['buyer_state', 'drug_name', 'transaction_date', 'calc_base_wt_in_gm']

# chunking
chunks = pd.read_csv(
    "../../data/DEA_2.csv.gz", 
    usecols=keep_cols, 
    chunksize=500000, 
    compression='gzip',
    low_memory=False
)

filtered_results = []

print("Starting extraction...")
for i, chunk in enumerate(chunks):
    mask = chunk['drug_name'].str.upper().isin(['OXYCODONE', 'HYDROCODONE'])
    small_chunk = chunk[mask]
    
    filtered_results.append(small_chunk)
    
    if i % 10 == 0:
        print(f"Processed {i * 500000} rows...")

# save
if filtered_results:
    df_mini = pd.concat(filtered_results)
    df_mini.to_csv('../../data/dea_summary_mini.csv', index=False)
    print(f"Done! Saved {len(df_mini)} rows to 'dea_summary_mini.csv'.")
else:
    print("No data found matching those drug names.")

Starting extraction...
Processed 0 rows...
Processed 5000000 rows...
Processed 10000000 rows...
Processed 15000000 rows...
Processed 20000000 rows...
Processed 25000000 rows...
Processed 30000000 rows...
Processed 35000000 rows...
Processed 40000000 rows...
Processed 45000000 rows...
Processed 50000000 rows...
Processed 55000000 rows...
Processed 60000000 rows...
Processed 65000000 rows...
Processed 70000000 rows...
Processed 75000000 rows...
Processed 80000000 rows...
Processed 85000000 rows...
Processed 90000000 rows...
Processed 95000000 rows...
Processed 100000000 rows...
Processed 105000000 rows...
Processed 110000000 rows...
Processed 115000000 rows...
Processed 120000000 rows...
Processed 125000000 rows...
Processed 130000000 rows...
Processed 135000000 rows...
Processed 140000000 rows...
Processed 145000000 rows...
Processed 150000000 rows...
Processed 155000000 rows...
Processed 160000000 rows...
Processed 165000000 rows...
Processed 170000000 rows...
Processed 175000000 rows.

In [2]:
# inspect the mini file
df = pd.read_csv('../../data/dea_summary_mini.csv')

print(df.info())
df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 360395927 entries, 0 to 360395926
Data columns (total 4 columns):
 #   Column              Dtype  
---  ------              -----  
 0   transaction_date    object 
 1   calc_base_wt_in_gm  float64
 2   buyer_state         object 
 3   drug_name           object 
dtypes: float64(1), object(3)
memory usage: 10.7+ GB
None


,transaction_date,calc_base_wt_in_gm,buyer_state,drug_name
0,01/01/2006,0.572708,NJ,HYDROCODONE
1,01/01/2006,1.432305,FL,HYDROCODONE
2,01/01/2006,1.432305,TX,HYDROCODONE
3,01/01/2006,1.432305,TX,HYDROCODONE
4,01/01/2006,2.864610,TX,HYDROCODONE


In [3]:
# Convert to format that can merge with NCHS

# get transaction year
print("Extracting years...")
df['year'] = pd.to_datetime(df['transaction_date']).dt.year

# grouping
print("Summing grams by state and year...")
df_summed = df.groupby(['buyer_state', 'year', 'drug_name'])['calc_base_wt_in_gm'].sum().reset_index()

# pivot so drugs are their own columns
print("Pivoting drug columns...")
df_pivot = df_summed.pivot_table(
    index=['buyer_state', 'year'], 
    columns='drug_name', 
    values='calc_base_wt_in_gm'
).reset_index()

# clean col names
df_pivot.columns.name = None  # Remove the drug_name label from columns
df_pivot = df_pivot.rename(columns={
    'buyer_state': 'state',
    'HYDROCODONE': 'hydro_gms',
    'OXYCODONE': 'oxy_gms'
})

# fill NAs with 0
df_pivot = df_pivot.fillna(0)

# Save final version
df_pivot.to_csv('../../data/dea_final.csv', index=False)
print("Transformation complete! Your file 'arcos_final_summary_06_14.csv' is ready for modeling.")
print(df_pivot.head())

Extracting years...
Summing grams by state and year...
Pivoting drug columns...
Transformation complete! Your file 'arcos_final_summary_06_14.csv' is ready for modeling.
  state  year     hydro_gms        oxy_gms
0    AK  2006  73936.970520  132344.307041
1    AK  2007  80968.424733  141050.065458
2    AK  2008  86069.808517  153192.816393
3    AK  2009  89413.086368  168255.904702
4    AK  2010  90779.457730  174587.019594
